# Seal detector v1 — Google Colab

Train a YOLO26 nano detector on the leakage-free 42,290-image seal dataset, evaluate the held-out test split, and export ONNX. Select a GPU runtime and run every cell in order.

In [ ]:
from pathlib import Path

RESUME = False
MODEL = 'yolo26n.pt'
RUN_NAME = 'seal-detector-v1'
EPOCHS = 100
IMAGE_SIZE = 960
SEED = 20260918

DRIVE_ROOT = Path('/content/drive/MyDrive/seal-ocr-detector')
DATASET_ZIP = DRIVE_ROOT / 'seal_detector_dataset.zip'
DATASET = Path('/content/seal_detector_dataset')
RUNS = DRIVE_ROOT / 'runs'
RUN_DIR = RUNS / RUN_NAME

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.155'], check=True)

import torch, ultralytics
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
gpu = torch.cuda.get_device_properties(0)
print({'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
       'gpu': gpu.name, 'vram_gb': round(gpu.total_memory / 2**30, 1)})

In [ ]:
import zipfile

assert DATASET_ZIP.is_file(), f'Upload the dataset ZIP to {DATASET_ZIP}'
if not DATASET.is_dir():
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        archive.extractall('/content')

expected = {'train': 29804, 'val': 8522, 'test': 3964}
for split, count in expected.items():
    images = list((DATASET / 'images' / split).glob('*.jpg'))
    labels = list((DATASET / 'labels' / split).glob('*.txt'))
    assert len(images) == len(labels) == count, (split, len(images), len(labels))

DATA_YAML = DATASET / 'data_colab.yaml'
DATA_YAML.write_text(f'''path: {DATASET}
train: images/train
val: images/val
test: images/test

names:
  0: seal
''')
print({'dataset': str(DATASET), 'total': sum(expected.values()), **expected})

In [ ]:
from ultralytics import YOLO

RUNS.mkdir(parents=True, exist_ok=True)
last = RUN_DIR / 'weights' / 'last.pt'
if RESUME:
    assert last.is_file(), f'No checkpoint to resume: {last}'
    model = YOLO(last)
    train_result = model.train(resume=True)
else:
    assert not RUN_DIR.exists(), f'Run already exists: {RUN_DIR}; set RESUME=True to continue it'
    model = YOLO(MODEL)
    train_result = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=-1,
        device=0, workers=2, amp=True, patience=15, seed=SEED,
        project=str(RUNS), name=RUN_NAME, plots=True,
    )

In [ ]:
import json, shutil
from datetime import datetime, timezone

best_path = RUN_DIR / 'weights' / 'best.pt'
assert best_path.is_file(), best_path
best = YOLO(best_path)
metrics = best.val(data=str(DATA_YAML), split='test', imgsz=IMAGE_SIZE, device=0)
onnx_path = Path(best.export(format='onnx', imgsz=IMAGE_SIZE, simplify=True, device=0))

artifacts = RUN_DIR / 'artifacts'
artifacts.mkdir(exist_ok=True)
for path in [best_path, RUN_DIR / 'weights' / 'last.pt', onnx_path,
             RUN_DIR / 'args.yaml', RUN_DIR / 'results.csv']:
    if path.is_file():
        shutil.copy2(path, artifacts / path.name)

metadata = {
    'modelVersion': RUN_NAME, 'baseModel': MODEL,
    'createdAtUtc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'datasetImages': 42290, 'splits': {'train': 29804, 'val': 8522, 'test': 3964},
    'excludedBenchmarkMatches': 414, 'imageSize': IMAGE_SIZE, 'epochs': EPOCHS,
    'seed': SEED, 'ultralyticsVersion': ultralytics.__version__,
    'torchVersion': torch.__version__, 'gpu': gpu.name,
    'testMetrics': {key: float(value) for key, value in metrics.results_dict.items()},
}
(artifacts / 'metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
archive = shutil.make_archive(str(RUN_DIR / RUN_NAME), 'zip', artifacts)
print({'best': str(best_path), 'onnx': str(onnx_path), 'package': archive,
       'test_metrics': metadata['testMetrics']})